In [1]:
from konlpy.tag import Okt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler
from collections import defaultdict
import numpy as np
from gensim.models import Word2Vec
import fasttext.util
import fasttext
import fasttext.util
import gzip
import os
import pandas as pd
import anthropic
from typing import List, Dict, Any
import json
import asyncio
import re
from tqdm.asyncio import tqdm_asyncio
import nest_asyncio
from datetime import datetime
import sys
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning) # FutureWarning 제거

In [2]:
df = pd.read_excel('../data/centum_data/21.11-24.6환자 CC_PI_치료계획.xlsx')
df['날짜'] = pd.to_datetime(df['날짜'])
df = df.iloc[:,1:]

api = pd.read_csv('../data/info.csv')
api_key = api.loc[0][1]

In [34]:
import os
import json
import logging
from datetime import datetime
from typing import List, Dict, Optional
import asyncio
import pandas as pd
import anthropic
from tenacity import retry, stop_after_attempt, wait_exponential
import re
from tqdm.asyncio import tqdm as tqdm_asyncio
import nest_asyncio
nest_asyncio.apply()  # Apply the patch for nested event loops


# Disable SettingWithCopyWarning
pd.options.mode.chained_assignment = None
# processed_df = asyncio.run(process_medical_data(df_sample, api_key))
# Configuration class for better organization
class Config:
    MODEL_NAME = "claude-3-5-haiku-20241022"
    MAX_TOKENS = 4096
    TEMPERATURE = 0
    BATCH_SIZE = 50
    SEMAPHORE_LIMIT = 5
    MAX_RETRIES = 3
    CHECKPOINT_DIR = "checkpoints"
    LOG_FILE = "medical_classifier.log"

# Improved logging setup
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(name)s - %(levelname)s - %(message)s",
    handlers=[
        logging.FileHandler(Config.LOG_FILE),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)

class CheckpointManager:
    """Checkpoint management class"""
    def __init__(self, checkpoint_dir: str = Config.CHECKPOINT_DIR):
        self.checkpoint_dir = checkpoint_dir
        os.makedirs(self.checkpoint_dir, exist_ok=True)

    def get_checkpoint_path(self, column: str) -> str:
        return os.path.join(self.checkpoint_dir, f"{column}_checkpoint.parquet")

    def save_checkpoint(self, df: pd.DataFrame, column: str) -> None:
        try:
            df.to_parquet(self.get_checkpoint_path(column))
            logger.info(f"Checkpoint saved for column {column}")
        except Exception as e:
            logger.error(f"Failed to save checkpoint for {column}: {str(e)}")

    def load_checkpoint(self, column: str) -> Optional[pd.DataFrame]:
        path = self.get_checkpoint_path(column)
        if os.path.exists(path):
            try:
                return pd.read_parquet(path)
            except Exception as e:
                logger.error(f"Failed to load checkpoint for {column}: {str(e)}")
        return None

class MedicalTextClassifier:
    def __init__(self, api_key: str):
        self.client = anthropic.Anthropic(api_key=api_key)
        self.semaphore = asyncio.Semaphore(Config.SEMAPHORE_LIMIT)
        self.checkpoint = CheckpointManager()
        self.classifiers = {
            'CC': self._classify_cc,
            '약': self._classify_medication,
            '장치': self._classify_device,
            '습관': self._classify_habit,
            '찜질': self._classify_hot_pack,
            '마사지, 스트레칭': self._classify_massage,
            'PI': self._classify_present_illness
        }

    async def process_all_columns(self, df: pd.DataFrame) -> pd.DataFrame:
        """Process all columns with improved error handling and checkpointing"""
        for column in self.classifiers.keys():
            if column in df.columns:
                logger.info(f"Processing column: {column}")
                df = await self._process_column_with_checkpoint(df, column)
        return df

    async def _process_column_with_checkpoint(self, df: pd.DataFrame, column: str) -> pd.DataFrame:
        """Process column with checkpoint support"""
        try:
            # Check for existing checkpoint
            checkpoint_df = self.checkpoint.load_checkpoint(column)
            if checkpoint_df is not None:
                df.update(checkpoint_df)
                logger.info(f"Resumed from checkpoint for {column}")
                return df

            # Process valid texts
            mask = df[column].notna() & df[column].str.strip().astype(bool)
            if not mask.any():
                return df

            texts_with_idx = [(idx, text) for idx, text in df.loc[mask, column].items()]
            dates_with_idx = [(idx, text) for idx, text in df.loc[mask, '날짜'].items()]

            print(texts_with_idx)
            print(dates_with_idx)

            results = await self._safe_process_batches(
                texts=[text for _, text in texts_with_idx],
                original_indices=[idx for idx, _ in texts_with_idx],
                classifier=self.classifiers[column],
                column=column
            )

            if results:
                result_df = pd.DataFrame(results).set_index('index')
                for col in result_df.columns:
                    new_col = f"{column}_{col}"
                    df[new_col] = result_df[col]

            self._cleanup_checkpoint(column)
            return df

        except Exception as e:
            logger.error(f"Critical error processing {column}: {str(e)}")
            raise

    async def _safe_process_batches(self, texts: List[str], original_indices: List[int],
                                  classifier, column: str) -> List[Dict]:
        """Process batches safely with retries and checkpointing"""
        results = []
        batch_size = Config.BATCH_SIZE

        for i in range(0, len(texts), batch_size):
            batch_texts = texts[i:i+batch_size]
            batch_indices = original_indices[i:i+batch_size]

            try:
                batch_results = await self._process_with_retry(
                    classifier, batch_texts, batch_indices
                )
                results.extend(batch_results)

                # Save partial results
                partial_df = pd.DataFrame(batch_results).set_index('index')
                self.checkpoint.save_checkpoint(partial_df, column)

            except Exception as e:
                logger.error(f"Batch {i//batch_size} failed: {str(e)}")
                continue

        return results

    @retry(stop=stop_after_attempt(Config.MAX_RETRIES),
           wait=wait_exponential(multiplier=1, min=2, max=10))
    async def _process_with_retry(self, classifier, batch_texts: List[str],
                                batch_indices: List[int]) -> List[Dict]:
        """Process with retry logic"""
        async with self.semaphore:
            results = await classifier(batch_texts, self.semaphore)
            return [{"index": idx, **res} for idx, res in zip(batch_indices, results)]

    def _cleanup_checkpoint(self, column: str) -> None:
        """Clean up checkpoint after successful processing"""
        try:
            checkpoint_path = self.checkpoint.get_checkpoint_path(column)
            if os.path.exists(checkpoint_path):
                os.remove(checkpoint_path)
                logger.info(f"Checkpoint cleaned up for {column}")
        except Exception as e:
            logger.error(f"Failed to cleanup checkpoint for {column}: {str(e)}")

    @retry(stop=stop_after_attempt(Config.MAX_RETRIES),
           wait=wait_exponential(multiplier=1, min=2, max=10))
    async def _make_api_call(self, prompt: str, semaphore: asyncio.Semaphore) -> List[Dict]:
        """Improved API call with better error handling"""
        try:
            async with semaphore:
                response = await asyncio.to_thread(
                    self.client.messages.create,
                    model=Config.MODEL_NAME,
                    max_tokens=Config.MAX_TOKENS,
                    temperature=Config.TEMPERATURE,
                    system="JSON 형식으로 응답하세요.",
                    messages=[{"role": "user", "content": prompt}]
                )

                content = response.content[0].text
                logger.debug(f"API Response: {content[:200]}...")

                result = self._validate_and_parse_json(content)
                if not result:
                    raise ValueError("Invalid JSON structure")
                return result

        except Exception as e:
            logger.error(f"API call failed: {str(e)}")
            raise

    def _validate_and_parse_json(self, content: str) -> List[Dict]:
        """Validate and parse JSON response"""
        try:
            # Extract JSON array using regex for more robust parsing
            array_pattern = r'\[(?:[^[\]]*|\[(?:[^[\]]*|\[[^[\]]*\])*\])*\]'
            matches = list(re.finditer(array_pattern, content))

            if not matches:
                return []

            longest_match = max(matches, key=lambda match: len(match.group()))
            potential_json = longest_match.group()

            parsed = json.loads(potential_json)
            if isinstance(parsed, list):
                # Iterate through each item in the list
                for item in parsed:
                    # Remove finish_reason key if present
                    item.pop('finish_reason', None)
                    # Recursively process nested objects
                    for key, value in item.items():
                        if isinstance(value, dict):  # Check if the value is a dictionary
                            value.pop('finish_reason', None)  # Remove key from nested object

                return parsed

            return []  # Return empty list if parsing fails

        except json.JSONDecodeError:
            logger.error(f"JSON parsing failed. Response content: {content[:500]}")
            return []

    # Original classifier methods remain the same but with improved error handling
    async def _classify_cc(self, texts: List[str], semaphore: asyncio.Semaphore) -> List[Dict]:
        """Classify Chief Complaints"""
        prompt = f"""다음 주요 증상(CC) 텍스트들을 분석하여 JSON 형식으로 분류해주세요.
            각 텍스트에 대해 다음 정보를 추출해주세요:
            1. location: 통증/증상 위치 (문자열)
            2. pain_type: 통증/증상 종류 (문자열)
            3. severity: 통증/증상 강도 (1-5, 없으면 null)
            4. duration: 지속 기간 (명시된 경우만, 문자열)

            텍스트 목록:
            {texts}

            다음 JSON 형식으로 응답해주세요:
            [{{
                "text": "원본 텍스트",
                "location": "위치",
                "pain_type": "통증 종류",
                "severity": "숫자 또는 null",
                "duration": "기간 또는 null"
            }}]"""
        return await self._make_api_call(prompt, semaphore)

    async def _classify_medication(self, texts: List[str], semaphore: asyncio.Semaphore) -> List[Dict]:
        """약물 복용 분류"""
        prompt = f"""다음 약물 복용 관련 텍스트들을 분석하여 JSON 형식으로 분류해주세요.

            각 텍스트에 대해 다음 정보를 추출해주세요:
            1. medication_type: 약물 종류 (진통제/소염제/근이완제 등)
            2. frequency: 복용 빈도 ('regular': 정기적, 'occasional': 간헐적, 'none': 미복용)
            3. duration: 복용 기간 (명시된 경우만)
            4. compliance: 복약 순응도 ('good': 양호, 'fair': 보통, 'poor': 불량)

            텍스트 목록:
            {texts}

            다음 JSON 형식으로 응답해주세요:
            [{{
                "text": "원본 텍스트",
                "medication_type": "약물 종류",
                "frequency": "복용 빈도",
                "duration": "기간 또는 null",
                "compliance": "순응도"
            }}]"""

        return await self._make_api_call(prompt, semaphore)

    async def _classify_device(self, texts: List[str], semaphore: asyncio.Semaphore) -> List[Dict]:
        """장치 사용 분류"""
        prompt = f"""다음 장치 사용 관련 텍스트들을 분석하여 JSON 형식으로 분류해주세요.

            각 텍스트에 대해 다음 정보를 추출해주세요:
            1. device_type: 장치 종류
            2. usage_pattern: 사용 패턴 ('constant': 상시착용, 'partial': 부분착용, 'rare': 거의미착용)
            3. duration: 사용 기간
            4. compliance: 착용 순응도 ('good': 양호, 'fair': 보통, 'poor': 불량)

            텍스트 목록:
            {texts}

            다음 JSON 형식으로 응답해주세요:
            [{{
                "text": "원본 텍스트",
                "device_type": "장치 종류",
                "usage_pattern": "사용 패턴",
                "duration": "기간 또는 null",
                "compliance": "순응도"
            }}]"""

        return await self._make_api_call(prompt, semaphore)

    async def _classify_habit(self, texts: List[str], semaphore: asyncio.Semaphore) -> List[Dict]:
        """습관 분류"""
        prompt = f"""다음 습관 관련 텍스트들을 분석하여 JSON 형식으로 분류해주세요.

            각 텍스트에 대해 다음 정보를 추출해주세요:
            1. habit_type: 습관 종류 (이갈이/편측성저작 등)
            2. frequency: 빈도 ('high': 매일/자주, 'medium': 가끔, 'low': 거의없음)
            3. awareness: 인지여부 ('aware': 인지, 'unaware': 미인지)
            4. improvement: 개선여부 ('improved': 개선, 'unchanged': 유지, 'worsened': 악화)

            텍스트 목록:
            {texts}

            다음 JSON 형식으로 응답해주세요:
            [{{
                "text": "원본 텍스트",
                "habit_type": "습관 종류",
                "frequency": "빈도",
                "awareness": "인지여부",
                "improvement": "개선여부"
            }}]"""

        return await self._make_api_call(prompt, semaphore)

    async def _classify_hot_pack(self, texts: List[str], semaphore: asyncio.Semaphore) -> List[Dict]:
        """찜질 분류"""
        prompt = f"""다음 찜질 관련 텍스트들을 분석하여 JSON 형식으로 분류해주세요.

            각 텍스트에 대해 다음 정보를 추출해주세요:
            1. status: 찜질 시행 여부 (0: 미시행, 1: 시행)
            2. frequency: 시행 빈도 ('high': 매일/자주, 'medium': 주 2-3회, 'low': 주 1회 이하)
            3. duration: 시행 시간 (분 단위 정수, 명시되지 않은 경우 null)
            4. method: 찜질 방법 ('hot': 온찜질, 'cold': 냉찜질, 'both': 둘 다, null: 불명확)

            텍스트 목록:
            {texts}

            다음 JSON 형식으로 응답해주세요:
            [{{
                "text": "원본 텍스트",
                "status": 0 또는 1,
                "frequency": "빈도",
                "duration": 숫자 또는 null,
                "method": "방법"
            }}]"""

        return await self._make_api_call(prompt, semaphore)

    async def _classify_massage(self, texts: List[str], semaphore: asyncio.Semaphore) -> List[Dict]:
        """마사지/스트레칭 분류"""
        prompt = f"""다음 마사지/스트레칭 관련 텍스트들을 분석하여 JSON 형식으로 분류해주세요.

            각 텍스트에 대해 다음 정보를 추출해주세요:
            1. type: 종류 ('massage': 마사지, 'stretching': 스트레칭, 'both': 둘다)
            2. frequency: 시행 빈도 ('high': 매일/자주, 'medium': 주 2-3회, 'low': 주 1회 이하)
            3. duration: 시행 시간 (분 단위 정수, 명시되지 않은 경우 null)
            4. method: 방법 ('self': 자가, 'professional': 전문가, 'both': 둘다)

            텍스트 목록:
            {texts}

            다음 JSON 형식으로 응답해주세요:
            [{{
                "text": "원본 텍스트",
                "type": "종류",
                "frequency": "빈도",
                "duration": 숫자 또는 null,
                "method": "방법"
            }}]"""

        return await self._make_api_call(prompt, semaphore)

    async def _classify_present_illness(self, texts: List[str], semaphore: asyncio.Semaphore) -> List[Dict]:
        """현재 질환(PI) 분류"""
        prompt = f"""다음 현재 질환(PI) 관련 텍스트들을 분석하여 JSON 형식으로 분류해주세요.

            각 텍스트에 대해 다음 정보를 추출해주세요:
            1. onset: 증상 발현 시기
            2. pattern: 증상 양상 ('constant': 지속성, 'intermittent': 간헐성, 'progressive': 진행성)
            3. aggravating_factors: 악화 요인 (리스트)
            4. status: 현재 상태 ('improving': 호전중, 'unchanged': 유지, 'worsening': 악화)

            텍스트 목록:
            {texts}

            다음 JSON 형식으로 응답해주세요:
            [{{
                "text": "원본 텍스트",
                "onset": "발현 시기",
                "pattern": "증상 양상",
                "aggravating_factors": ["요인1", "요인2"],
                "status": "현재 상태"
            }}]"""

        return await self._make_api_call(prompt, semaphore)

async def process_medical_data(df: pd.DataFrame, api_key: str) -> pd.DataFrame:
    """Process medical data with comprehensive error handling and logging"""
    classifier = MedicalTextClassifier(api_key)
    start_time = datetime.now()
    logger.info(f"Starting medical data processing at {start_time}")

    try:
        # Process data
        processed_df = await classifier.process_all_columns(df)

        # Log statistics
        end_time = datetime.now()
        processing_time = end_time - start_time
        total_rows = len(df)
        processed_columns = [col for col in df.columns if col in classifier.classifiers]

        logger.info("=== Processing Summary ===")
        logger.info(f"Total time: {processing_time}")
        logger.info(f"Total rows processed: {total_rows}")
        logger.info(f"Columns processed: {processed_columns}")

        # Calculate success rates for each column
        for col in processed_columns:
            total_entries = df[col].notna().sum()
            processed_entries = sum(1 for col_name in processed_df.columns
                                 if col_name.startswith(f"{col}_")
                                 and processed_df[col_name].notna().any())
            success_rate = (processed_entries / total_entries * 100) if total_entries > 0 else 0
            logger.info(f"{col} - Success rate: {success_rate:.2f}%")

        return processed_df

    except Exception as e:
        logger.critical(f"Critical error during medical data processing: {str(e)}")
        raise
    finally:
        # Cleanup (without await)
        if classifier.client:  # Check if client is initialized
            classifier.client.close() # Run close() without await
        logger.info("Processing completed and resources cleaned up")

if __name__ == "__main__":
    # Setup logging
    logging.basicConfig(
        level=logging.INFO,
        format="%(asctime)s - %(name)s - %(levelname)s - %(message)s",
        handlers=[
            logging.FileHandler(Config.LOG_FILE),
            logging.StreamHandler()
        ]
    )
    logger = logging.getLogger(__name__)

    try:
        # Load sample data
        df_sample = df.head(10)

        # Set API key (should be in environment variable or config file in production)

        # Process data
        logger.info("Starting sample data processing")
        loop = asyncio.get_event_loop()
        processed_df = loop.run_until_complete(process_medical_data(df_sample, api_key))

        # Save results
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        output_file = f'processed_medical_data_{timestamp}.parquet'
        processed_df.to_parquet(output_file)
        logger.info(f"Data successfully saved to {output_file}")

        # Print basic statistics
        logger.info("\n=== Processing Results ===")
        for column in processed_df.columns:
            if '_' in column:  # Only show derived columns
                valid_count = processed_df[column].notna().sum()
                logger.info(f"{column}: {valid_count} valid entries")

                if processed_df[column].dtype in ['object', 'category']:
                    value_counts = processed_df[column].value_counts()
                    logger.info(f"Value distribution:\n{value_counts}\n")

    except Exception as e:
        logger.error(f"Main execution failed: {str(e)}")
        sys.exit(1)
    finally:
        logger.info("Program execution completed")



2025-02-16 17:16:29,858 - __main__ - INFO - Starting sample data processing
2025-02-16 17:16:29,889 - __main__ - INFO - Starting medical data processing at 2025-02-16 17:16:29.889273
2025-02-16 17:16:29,890 - __main__ - INFO - Processing column: CC


[(0, '예전엔 왼쪽통증과 입벌림이 힘들어서 서울대병원 30년 전 장치도 했었어요장치는 두고 왔어요 - 착용은 1년정도 하고 계속 괜찮다가 이번에 오른쪽 증상 생겼어요 저번주 토요일 아침에 하품하다가 오른쪽 턱이 쥐난거처럼 경직되더니 현재 치아맞물림도 오른쪽이 어금니가 뜨고 오른쪽 턱이 잘 안벌어져요. 그 날 바로 일반치과 갔고 손으로 턱 넣고 했는데 맞춰지지않는다고 구강내과 가보라고해서 왔어요. 통증은 없어요. 소리는 왼쪽에 있었는데 장치치료 받고 사라져서 현재까지 없어요. 이악무는습관없어요이갈이 어렸을 때 만 잠은 잘자요스트레스 딱히 없어요(현재 위 남아있는 유치 살짝 흔들리고 있는데 잇몸치료받았고 다니는 치과 있어요.) 초음파ok'), (1, '구강내과#2[도착]물리치료 , APS del증상: 오른쪽으로 식사 많이하면 턱이 불편했고  , 오른쪽 귀 만지면 앞에부분이 아팠어요, 턱벌어지는거는 비슷해요오른쪽 위 유치 언제 빼야할지, 임플란트 언제 할수있나요?치아 시린 부분이 있어요. 치아 패인 부분 체크 해주세요'), (2, '구강내과#3물리치료 , 장치 ck, 발즉(OP)증상: 오른쪽으로 씹으면 조금 아파요. 어제는 오른쪽 목까지 아파서 왼쪽으로만 씹었어요.      3일전부터 유치 안쪽 잇몸이 부었다가 오후되면 가라앉아요.      턱을 앞으로 내미는 느낌은 없어요.         자다보면 혀가 아래 앞니랑 장치 사이에 계속 끼어요.'), (3, '구강내과#4s/o, #34,35 ck 후 CA, 물리치료,장치ck증상: 오른쪽 씹을때 조금 아플때도 있고 불편할때도 있는거 저번이랑 비슷해요        음식 종류에따라 달라요 vas3        아침에 일어나서 장치를 빼고 나면 오른쪽 아랫턱이 뻐쩍찌근한 느낌 있다가        좀 지나면 사라져요         제가 저번주에 한번 푹 자고싶어서 하루 장치를 안끼고 잤는데        아침에 일어났을때 안아팠어요       예전에는 온찜질만 했었는데..        이번에는 염증있다고 냉찜질도 하라고 하셨

2025-02-16 17:16:48,726 - httpx - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2025-02-16 17:16:48,796 - __main__ - INFO - Checkpoint saved for column CC
2025-02-16 17:16:48,801 - __main__ - INFO - Checkpoint cleaned up for CC
2025-02-16 17:16:48,802 - __main__ - INFO - Processing column: 약


[(1, '약: 약먹고 나서부터 갈비뼈부터 등까지 근육이 아팠어요, 아직 옆구리 있는 부분이 아파요')]
[(1, Timestamp('2023-02-01 00:00:00'))]


2025-02-16 17:16:54,576 - httpx - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2025-02-16 17:16:54,595 - __main__ - INFO - Checkpoint saved for column 약
2025-02-16 17:16:54,599 - __main__ - INFO - Checkpoint cleaned up for 약
2025-02-16 17:16:54,600 - __main__ - INFO - Processing column: 습관


[(1, '습관: 딱딱하고 질긴거 피했어요. 이랑 이 안닿게 했어요'), (2, '습관: 딱딱하고 질긴 음식 안 먹으려고 노력을 해요, 치아끼리 닿지 않도록 해요.'), (3, '습관: 치아끼리 닿지 않게 했어요 딱딱하고 질긴거 피했어요'), (4, '습관: 딱딱하고 질긴거 피하려하지만 가끔 고기 덜질긴거 먹으려해요'), (5, '습관: 질기고 딱딱 피하고 최근에는 어금니 닿게 무는게 아니라 앞니가 닿아있는 느낌이 들어요.'), (6, '습관: 딱딱하고 질긴 음식 가끔 먹을 일이 있으면 먹어요. 치아끼리 닿지 않도록 해요.'), (7, '습관:고기조금먹었어요 치아끼리 안닿게 했어요/'), (8, '습관: 가끔 고기류 정도 먹었어요/ 치아끼리 안닿게 턱에 힘 풀고 지냇어요'), (9, '습관: 딱딱하고 질긴음식 가끔 먹어요(고기류,나물)')]
[(1, Timestamp('2023-02-01 00:00:00')), (2, Timestamp('2023-02-17 00:00:00')), (3, Timestamp('2023-03-21 00:00:00')), (4, Timestamp('2023-04-21 00:00:00')), (5, Timestamp('2023-05-19 00:00:00')), (6, Timestamp('2023-07-19 00:00:00')), (7, Timestamp('2023-09-19 00:00:00')), (8, Timestamp('2023-11-21 00:00:00')), (9, Timestamp('2023-12-12 00:00:00'))]


2025-02-16 17:17:03,987 - httpx - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2025-02-16 17:17:04,009 - __main__ - INFO - Checkpoint saved for column 습관
2025-02-16 17:17:04,020 - __main__ - INFO - Checkpoint cleaned up for 습관
2025-02-16 17:17:04,021 - __main__ - INFO - Processing column: 마사지, 스트레칭


[(9, '마사지,스트레칭: 매일 아침에')]
[(9, Timestamp('2023-12-12 00:00:00'))]


2025-02-16 17:17:07,672 - httpx - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2025-02-16 17:17:07,681 - __main__ - INFO - Checkpoint saved for column 마사지, 스트레칭
2025-02-16 17:17:07,685 - __main__ - INFO - Checkpoint cleaned up for 마사지, 스트레칭
2025-02-16 17:17:07,686 - __main__ - INFO - Processing column: PI
2025-02-16 17:17:07,688 - __main__ - INFO - === Processing Summary ===
2025-02-16 17:17:07,688 - __main__ - INFO - Total time: 0:00:37.798843
2025-02-16 17:17:07,689 - __main__ - INFO - Total rows processed: 10
2025-02-16 17:17:07,690 - __main__ - INFO - Columns processed: ['CC', '약', '습관', '마사지, 스트레칭', 'PI']
2025-02-16 17:17:07,693 - __main__ - INFO - CC - Success rate: 50.00%
2025-02-16 17:17:07,696 - __main__ - INFO - 약 - Success rate: 400.00%
2025-02-16 17:17:07,700 - __main__ - INFO - 습관 - Success rate: 55.56%
2025-02-16 17:17:07,703 - __main__ - INFO - 마사지, 스트레칭 - Success rate: 400.00%
2025-02-16 17:17:07,704 - __main__ - INFO - PI - Success 

In [35]:
processed_df.columns

Index(['환자번호', '날짜', 'CC', '약', '장치 ', '습관', '찜질 ', '마사지, 스트레칭', 'PI', 'CMO',
       'MMO', 'Cap.pal', 'M.pal', 'Noise', 'Loading', 'Occlusion', 'OJ/OB',
       'Class', 'Midline Shift', 'Deviation', 'CR-CO', 'Tongue ridging',
       'Mucosal ridging', 'Ultrasono', 'Rt', 'Lt',
       'Lateral excursion Protrusive excursion', 'End feel', '치료계획',
       'T-scan 악화/개선', 'CBCT 악화/개선', 'CBCT 판독소견', 'CC_text', 'CC_location',
       'CC_pain_type', 'CC_severity', 'CC_duration', '약_text',
       '약_medication_type', '약_frequency', '약_duration', '약_compliance',
       '습관_text', '습관_habit_type', '습관_frequency', '습관_awareness',
       '습관_improvement', '마사지, 스트레칭_text', '마사지, 스트레칭_type',
       '마사지, 스트레칭_frequency', '마사지, 스트레칭_duration', '마사지, 스트레칭_method'],
      dtype='object')

In [31]:
processed_df.columns

Index(['환자번호', '날짜', 'CC', '약', '장치 ', '습관', '찜질 ', '마사지, 스트레칭', 'PI', 'CMO',
       'MMO', 'Cap.pal', 'M.pal', 'Noise', 'Loading', 'Occlusion', 'OJ/OB',
       'Class', 'Midline Shift', 'Deviation', 'CR-CO', 'Tongue ridging',
       'Mucosal ridging', 'Ultrasono', 'Rt', 'Lt',
       'Lateral excursion Protrusive excursion', 'End feel', '치료계획',
       'T-scan 악화/개선', 'CBCT 악화/개선', 'CBCT 판독소견', 'CC_text', 'CC_location',
       'CC_pain_type', 'CC_severity', 'CC_duration', '약_text',
       '약_medication_type', '약_frequency', '약_duration', '약_compliance',
       '습관_text', '습관_habit_type', '습관_frequency', '습관_awareness',
       '습관_improvement'],
      dtype='object')

In [39]:
processed_df[['환자번호', '날짜',
       'CC_location','CC_pain_type','CC_severity', 'CC_duration', 
       '약_medication_type', '약_frequency', '약_duration', '약_compliance',
        '습관_habit_type', '습관_frequency', '습관_awareness','습관_improvement',
       '마사지, 스트레칭_type','마사지, 스트레칭_frequency', '마사지, 스트레칭_duration', '마사지, 스트레칭_method'
        ]]

,환자번호,날짜,CC_location,CC_pain_type,CC_severity,CC_duration,약_medication_type,약_frequency,약_duration,약_compliance,습관_habit_type,습관_frequency,습관_awareness,습관_improvement,"마사지, 스트레칭_type","마사지, 스트레칭_frequency","마사지, 스트레칭_duration","마사지, 스트레칭_method"
0,2301-01,2023-01-17,오른쪽 턱,"턱 경직, 치아맞물림 이상",NaN,저번주 토요일 아침부터,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2301-01,2023-02-01,"오른쪽 턱, 오른쪽 귀","턱 불편감, 귀 앞부분 통증",NaN,None,진통제/소염제,occasional,None,fair,편측성저작,low,aware,improved,NaN,NaN,NaN,NaN
2,2301-01,2023-02-17,"오른쪽 턱, 목","씹을 때 통증, 잇몸 부종",2.0,3일전부터,NaN,NaN,NaN,NaN,편측성저작,low,aware,improved,NaN,NaN,NaN,NaN
3,2301-01,2023-03-21,"오른쪽 턱, 오른쪽 아랫턱, 잇몸","씹을 때 통증, 아랫턱 뻐근함",3.0,None,NaN,NaN,NaN,NaN,편측성저작,low,aware,improved,NaN,NaN,NaN,NaN
4,2301-01,2023-04-21,오른쪽 턱,씹을 때 통증,NaN,None,NaN,NaN,NaN,NaN,편측성저작,medium,aware,unchanged,NaN,NaN,NaN,NaN
5,2301-01,2023-05-19,오른쪽 턱,씹을 때 간헐적 통증,1.0,None,NaN,NaN,NaN,NaN,이갈이,medium,aware,worsened,NaN,NaN,NaN,NaN
6,2301-01,2023-07-19,"왼쪽 턱, 왼쪽 아래 치아",치아 불편감,NaN,None,NaN,NaN,NaN,NaN,편측성저작,medium,aware,unchanged,NaN,NaN,NaN,NaN
7,2301-01,2023-09-19,None,None,NaN,None,NaN,NaN,NaN,NaN,편측성저작,low,aware,improved,NaN,NaN,NaN,NaN
8,2301-01,2023-11-21,어금니,씹을 때 소음,NaN,10일전부터,NaN,NaN,NaN,NaN,편측성저작,medium,aware,unchanged,NaN,NaN,NaN,NaN
9,2301-01,2023-12-12,오른쪽 어금니,어금니 접촉 이상,NaN,None,NaN,NaN,NaN,NaN,편측성저작,medium,aware,unchanged,both,high,None,self


In [5]:
df.CMO

0          34mm --> mm after spray and stretch
1          38mm --> mm after spray and stretch
2          38mm --> mm after spray and stretch
3          40mm --> mm after spray and stretch
4          48mm --> mm after spray and stretch
                         ...                  
28103    20mm --> 53mm after spray and stretch
28104        mm --> mm after spray and stretch
28105      29mm --> mm after spray and stretch
28106      20mm --> mm after spray and stretch
28107      52mm --> mm after spray and stretch
Name: CMO, Length: 28108, dtype: object

In [19]:
df['Noise'].sample(10)

20631           Lt click 아직 있음
7330                         -
4118     Lt) click Rt) popping
17703                      n/s
554                          -
13483              both) click
14425                        -
22501                      NaN
15212                      NaN
6175                         -
Name: Noise, dtype: object